# Exercise 12.1: Containerize an analysis

From *Programming in High Energy Particle Physics*, Chapter 12

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/hep-programming-guide-notebooks/blob/main/ch12/ex12_1_solution.ipynb)

Exercise 12.1 Containerize an analysis Containerize a simple HEP analysis using Docker. Create a Dockerfile that installs ROOT, Python, and your analysis code, and produces plots as output when run.

<details><summary>Hint</summary>

Do not build ROOT yourself: start from an official ROOT image, e.g. FROM rootproject/root:latest (pin a specific version tag for reproducibility), which already contains ROOT and its Python bindings. Add RUN pip install uproot awkward hist mplhep , COPY your code, and set CMD to run the analysis. Mount an output directory with docker run -v "$PWD/plots:/work/plots" ... so the plots appear on the host. On a cluster without Docker, the same image runs with apptainer run docker://... .

</details>

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    get_ipython().run_line_magic("pip", "install -q numpy matplotlib")

**Colab limitation:** Colab does not provide a Docker daemon. The notebook writes the complete Dockerfile and analysis script, then runs the script directly for a lightweight check. Container execution is local or on an Apptainer cluster.

### Step 1: Write a runnable analysis script

In [ ]:
from pathlib import Path
analysis = r"""import os
import numpy as np
import matplotlib.pyplot as plt
rng=np.random.default_rng(42)
pt=rng.exponential(35,10000)
os.makedirs('plots',exist_ok=True)
fig,ax=plt.subplots(figsize=(7,4))
ax.hist(pt,bins=np.linspace(0,200,41),histtype='step')
ax.set(xlabel='Toy particle transverse momentum [GeV]',ylabel='Events / 5 GeV')
fig.tight_layout(); fig.savefig('plots/pt.png'); plt.close(fig)
print('Wrote plots/pt.png')
"""
Path('analysis.py').write_text(analysis)

### Step 2: Write a pinned ROOT Dockerfile

In [ ]:
dockerfile="""FROM rootproject/root:6.32.04-ubuntu24.04
WORKDIR /work
RUN apt-get update && apt-get install -y --no-install-recommends python3-pip && rm -rf /var/lib/apt/lists/*
RUN python3 -m pip install --break-system-packages --no-cache-dir uproot awkward hist mplhep matplotlib numpy
COPY analysis.py /work/analysis.py
CMD ["python3", "/work/analysis.py"]
"""
Path('Dockerfile').write_text(dockerfile)
print(dockerfile)

### Step 3: Run the analysis directly in Colab and show the plot

In [ ]:
import subprocess,sys
subprocess.run([sys.executable,'analysis.py'],check=True)
from IPython.display import display,Image
display(Image(filename='plots/pt.png'))
print('Local: docker build -t hep-toy:6.32 .')
print('Local: docker run --rm -v "$PWD/plots:/work/plots" hep-toy:6.32')
print('Cluster: apptainer run --bind "$PWD/plots:/work/plots" docker://<your-published-image>')

The image starts from a pinned official ROOT build and the script writes a labelled toy distribution to a mounted plots directory. Colab executes the analysis directly; Docker build/run must be performed on a machine with Docker. For Apptainer, use a published copy of the same image.